# RankGen graph-generator experiment: GCDG

This notebook uses the split you described:

- `train1`: real graphs used to fit the generator.
- `generated`: positive and negative generated graphs sampled from the generator fitted on `train1`.
- `train2`: real reference graphs, not used by the generator.
- `test`: held-out labelled graphs for the downstream classification task.

The train1 vs. train1+train2 classifier check is a sanity check for RankGen utility. Utility asks whether adding generated graphs helps as much as adding real reference graphs. If `train1 + train2` is not at least five percentage points better than `train1`, the denominator is too small and the utility score becomes noisy or uninformative.


In [ ]:
# ===== RankGen split/evaluation configuration =====
from pathlib import Path

SEED = 42

# All three generator notebooks should reuse this exact split.
# First notebook run creates it; later notebooks load it.
ARTIFACT_DIR = Path("artifacts/rankgen")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
SHARED_SPLIT_PATH = ARTIFACT_DIR / "shared_rankgen_train1_train2_test.pkl"
SHARED_GAP_TABLE_PATH = ARTIFACT_DIR / "shared_rankgen_reference_gap.csv"
REUSE_SHARED_SPLIT_IF_EXISTS = True
OVERWRITE_SHARED_SPLIT = False

# Use an in-memory dataset if `graphs` and `targets` already exist.
# Otherwise set DATASET_PICKLE to a pickle containing (graphs, targets) or {'graphs': ..., 'targets': ...}.
# These are needed only when the shared split file does not exist or OVERWRITE_SHARED_SPLIT=True.
DATASET_PICKLE = None
TARGETS_PICKLE = None

# Dataset source for a fresh notebook kernel.
# Default is a labelled binary graph task: positive=cycle target, negative=tree target.
# Set DATASET_CHOICE="custom" and DATASET_PICKLE/TARGETS_PICKLE if you want your own data.
DATASET_CHOICE = "artificial_binary"
ARTIFICIAL_N_PER_CLASS = 500
ARTIFICIAL_ALPHABET_SIZE = 30
ARTIFICIAL_TARGET_SIZE = 5
ARTIFICIAL_CONTEXT_SIZE = 5
ARTIFICIAL_LINK_EDGES = 1
ARTIFICIAL_POSITIVE_TARGET_TYPE = "cycle"
ARTIFICIAL_POSITIVE_CONTEXT_TYPE = "cycle"
ARTIFICIAL_NEGATIVE_TARGET_TYPE = "tree"
ARTIFICIAL_NEGATIVE_CONTEXT_TYPE = "cycle"
ARTIFICIAL_CANONICALISE = True

TEST_SIZE = 0.20
CANDIDATE_TRAIN1_FRACTIONS = (0.50, 0.40, 0.33, 0.25, 0.20, 0.15, 0.10)
MIN_REFERENCE_GAIN = 0.05
GAP_METRIC = "macro_f1"
GAP_REPEATS = 3

# Same vectorizer for every generator: NSPPK, fitted on the same real reference side.
NSPPK_PARAMS = dict(radius=2, distance=4, connector=0, nbits=12, dense=False, parallel=False)
RANKGEN_FIT_SCOPE = "train1_train2"  # fit NSPPK on train1+train2, transform generated and test

RANKGEN_N_ITER = 30          # increase to 100+ for final reporting
RANKGEN_FRACTION = 0.80
RANKGEN_ESTIMATORS = 300
RANKGEN_SCORE_METRIC = "macro_f1"
RANKGEN_PARALLEL = False
RANKGEN_VERBOSE = 1


In [ ]:
# ===== GCDG generation configuration =====
GENERATOR_KEY = "gcdg"
GENERATOR_NAME = "GCDG"

# If you already generated graphs, set this to the pickle payload path.
GENERATED_PAYLOAD_PATH = None
USE_EXISTING_GENERATED_VARIABLES = False

# The notebook will look for one of these fitted model variables if no payload is provided.
GCDG_MODEL_VARIABLE_NAMES = ("decompositional_encoder_decoder", "gcdg", "denoise")

# `seeded` decodes from train1 condition/reference graphs class-by-class.
# Use `prior` only if you want unconditional/prior sampling from the trained GCDG model.
DENOISE_GENERATION_MODE = "seeded"


In [ ]:
# ===== Load/create the shared RankGen split =====
from collections import Counter
import pandas as pd
from IPython.display import display

from rankgen_graph_experiment import (
    add_rank_columns,
    default_classifier_factory,
    default_nsppk_factory,
    generation_counts_from_targets,
    load_graphs_targets_pickle,
    load_rankgen_split,
    resolve_rankgen_dataset,
    make_rankgen_split_with_gap,
    run_rankgen_for_generated,
    save_graphs_targets_pickle,
    save_rankgen_split,
    write_results,
)

vectorizer_factory = default_nsppk_factory(**NSPPK_PARAMS)
classifier_factory = default_classifier_factory(seed=SEED, n_estimators=RANKGEN_ESTIMATORS)

should_load_shared = (
    REUSE_SHARED_SPLIT_IF_EXISTS
    and not OVERWRITE_SHARED_SPLIT
    and SHARED_SPLIT_PATH.exists()
)

if should_load_shared:
    rankgen_split = load_rankgen_split(SHARED_SPLIT_PATH)
    if SHARED_GAP_TABLE_PATH.exists():
        rankgen_gap_table = pd.read_csv(SHARED_GAP_TABLE_PATH)
    else:
        rankgen_gap_table = pd.DataFrame([{
            "train1_fraction": rankgen_split.selected_train1_fraction,
            "train1_score": rankgen_split.train1_score,
            "train1_plus_train2_score": rankgen_split.train1_plus_train2_score,
            "reference_gain": rankgen_split.reference_gain,
        }])
    print("loaded shared split:", SHARED_SPLIT_PATH.resolve())
else:
    graphs_in_kernel = globals().get("graphs", None)
    targets_in_kernel = globals().get("targets", None)
    use_kernel_dataset = (
        DATASET_PICKLE is None
        and graphs_in_kernel is not None
        and targets_in_kernel is not None
        and str(DATASET_CHOICE).strip().lower() in {"kernel", "in_memory", "memory"}
    )
    graphs, targets = resolve_rankgen_dataset(
        dataset_choice=DATASET_CHOICE,
        dataset_pickle=DATASET_PICKLE,
        targets_pickle=TARGETS_PICKLE,
        graphs=graphs_in_kernel if use_kernel_dataset else None,
        targets=targets_in_kernel if use_kernel_dataset else None,
        artificial_n_per_class=ARTIFICIAL_N_PER_CLASS,
        artificial_alphabet_size=ARTIFICIAL_ALPHABET_SIZE,
        artificial_target_size=ARTIFICIAL_TARGET_SIZE,
        artificial_context_size=ARTIFICIAL_CONTEXT_SIZE,
        artificial_link_edges=ARTIFICIAL_LINK_EDGES,
        artificial_positive_target_type=ARTIFICIAL_POSITIVE_TARGET_TYPE,
        artificial_positive_context_type=ARTIFICIAL_POSITIVE_CONTEXT_TYPE,
        artificial_negative_target_type=ARTIFICIAL_NEGATIVE_TARGET_TYPE,
        artificial_negative_context_type=ARTIFICIAL_NEGATIVE_CONTEXT_TYPE,
        artificial_canonicalise=ARTIFICIAL_CANONICALISE,
    )

    print("dataset choice:", DATASET_CHOICE)
    print("dataset graphs:", len(graphs))
    print("class counts:", Counter(targets))

    rankgen_split, rankgen_gap_table = make_rankgen_split_with_gap(
        graphs,
        targets,
        test_size=TEST_SIZE,
        candidate_train1_fractions=CANDIDATE_TRAIN1_FRACTIONS,
        min_reference_gain=MIN_REFERENCE_GAIN,
        seed=SEED,
        vectorizer_factory=vectorizer_factory,
        classifier_factory=classifier_factory,
        metric=GAP_METRIC,
        repeats=GAP_REPEATS,
        raise_on_fail=True,
    )
    save_rankgen_split(rankgen_split, SHARED_SPLIT_PATH)
    rankgen_gap_table.to_csv(SHARED_GAP_TABLE_PATH, index=False)
    print("saved shared split:", SHARED_SPLIT_PATH.resolve())

display(rankgen_gap_table)
print("selected train1_fraction:", rankgen_split.selected_train1_fraction)
print("train1/train2/test:", len(rankgen_split.train1_graphs), len(rankgen_split.train2_graphs), len(rankgen_split.test_graphs))
print("reference gain:", rankgen_split.reference_gain)
print("train1 counts:", Counter(rankgen_split.train1_targets))
print("train2 counts:", Counter(rankgen_split.train2_targets))
print("test counts:", Counter(rankgen_split.test_targets))
print("generated counts should match train1:", generation_counts_from_targets(rankgen_split.train1_targets))
print("RankGen vectorizer: NSPPK", NSPPK_PARAMS, "| fit_scope:", RANKGEN_FIT_SCOPE)

# Export aliases for model-training cells that expect these names.
train1_graphs = rankgen_split.train1_graphs
train1_targets = rankgen_split.train1_targets
train2_graphs = rankgen_split.train2_graphs
train2_targets = rankgen_split.train2_targets
test_graphs = rankgen_split.test_graphs
test_targets = rankgen_split.test_targets
train_whole_graphs = rankgen_split.train_whole_graphs
train_whole_targets = rankgen_split.train_whole_targets
train_graphs = train1_graphs
train_targets = train1_targets


In [ ]:
# ===== Load/use fitted GCDG and generate class counts matching train1 =====
from pathlib import Path
from rankgen_graph_experiment import generate_from_model, load_graphs_targets_pickle, save_graphs_targets_pickle

payload_path = Path(GENERATED_PAYLOAD_PATH) if GENERATED_PAYLOAD_PATH is not None else None

if payload_path is not None and payload_path.exists():
    generated_graphs, generated_targets = load_graphs_targets_pickle(payload_path)
elif USE_EXISTING_GENERATED_VARIABLES and "generated_graphs" in globals() and "generated_targets" in globals():
    generated_graphs = list(generated_graphs)
    generated_targets = list(generated_targets)
else:
    gcdg_model = None
    gcdg_model_name = None
    for name in GCDG_MODEL_VARIABLE_NAMES:
        if name in globals() and globals()[name] is not None:
            gcdg_model = globals()[name]
            gcdg_model_name = name
            break
    if gcdg_model is None:
        raise RuntimeError(
            "No fitted GCDG model found. Run your GCDG training cells first, set one of "
            f"{GCDG_MODEL_VARIABLE_NAMES}, or set GENERATED_PAYLOAD_PATH."
        )
    print("using fitted GCDG model variable:", gcdg_model_name)
    generated_graphs, generated_targets = generate_from_model(
        gcdg_model,
        model_kind="gcdg",
        train1_graphs=rankgen_split.train1_graphs,
        train1_targets=rankgen_split.train1_targets,
        seed=SEED,
        denoise_generation_mode=DENOISE_GENERATION_MODE,
    )

GENERATED_SAVE_PATH = ARTIFACT_DIR / "gcdg_generated_from_train1.pkl"
save_graphs_targets_pickle(generated_graphs, generated_targets, GENERATED_SAVE_PATH)
print("generated payload saved:", GENERATED_SAVE_PATH.resolve())


In [ ]:
# ===== Run RankGen for the generated graph set =====
from collections import Counter
import pandas as pd
from IPython.display import display

if "generated_graphs" not in globals() or "generated_targets" not in globals():
    raise RuntimeError("Create or load `generated_graphs` and `generated_targets` first.")

print("generated graphs:", len(generated_graphs))
print("generated class counts:", Counter(generated_targets))

rankgen_result = run_rankgen_for_generated(
    model_name=GENERATOR_NAME,
    generated_graphs=generated_graphs,
    generated_targets=generated_targets,
    split=rankgen_split,
    vectorizer_factory=vectorizer_factory,
    n_iterations=RANKGEN_N_ITER,
    fraction=RANKGEN_FRACTION,
    estimator_n=RANKGEN_ESTIMATORS,
    score_metric=RANKGEN_SCORE_METRIC,
    parallel=RANKGEN_PARALLEL,
    verbose=RANKGEN_VERBOSE,
    fit_scope=RANKGEN_FIT_SCOPE,
)

rankgen_results = add_rank_columns(pd.DataFrame([rankgen_result]))
display(rankgen_results)

RESULTS_PATH = ARTIFACT_DIR / f"{GENERATOR_KEY}_rankgen_results.csv"
write_results(rankgen_results, RESULTS_PATH)
print("saved:", RESULTS_PATH.resolve())


In [ ]:
# ===== Optional: combine VAE, DiGress, and GCDG result CSVs and rank them together =====
import pandas as pd
from IPython.display import display
from rankgen_graph_experiment import add_rank_columns, write_results

RESULT_FILES = [
    ARTIFACT_DIR / "vgae_rankgen_results.csv",
    ARTIFACT_DIR / "digress_rankgen_results.csv",
    ARTIFACT_DIR / "gcdg_rankgen_results.csv",
]
frames = []
for path in RESULT_FILES:
    if path.exists():
        frames.append(pd.read_csv(path))
    else:
        print("missing:", path)

if frames:
    combined_rankgen_results = add_rank_columns(pd.concat(frames, ignore_index=True))
    display(combined_rankgen_results)
    combined_path = ARTIFACT_DIR / "combined_rankgen_results.csv"
    write_results(combined_rankgen_results, combined_path)
    print("saved:", combined_path.resolve())
else:
    print("No result CSVs found yet.")
